In [1]:
import pandas as pd
import numpy as np

# 1. Carregar a base original (Morumbi)

In [2]:
# Carrega o arquivo de entrada (arquivo organizado em `data/filling_Ceps/`)
df = pd.read_csv('../../data/filling_Ceps/Elvira Brandão Morumbi - Euvira Brandão Dados ADS_coords_corrigidas_com_enderecos.csv')

# 2. Tratar a renda para número

In [3]:
import re

def parse_renda_seguro(x):
    if pd.isna(x):
        return np.nan
    x = str(x)

    # remove qualquer símbolo que não seja número, vírgula ou ponto
    x = re.sub(r"[^0-9,\.]", "", x)

    # caso venha no formato errado tipo "19.448.421.299.999.900"
    # mantemos somente o último grupo decimal
    if x.count(".") > 1:
        # remove TODOS os pontos; eles NÃO representam milhar
        x = x.replace(".", "")

    # agora troca vírgula por ponto
    x = x.replace(",", ".")

    try:
        return float(x)
    except:
        return np.nan


df["renda_media_num"] = df["renda_media"].apply(parse_renda_seguro)

# 3. Criar Total_0_9 (0–4 + 5–9 anos)

In [4]:
df["Total_0_9"] = (
    df["v01031_0_4anos"].fillna(0) +
    df["v01032_5_9anos"].fillna(0)
)

# 4. Corrigir renda para 2025 (inflação)

In [5]:
inflation_factor = 1.155
df["renda_atualizada_2025"] = df["renda_media_num"] * inflation_factor

# 5. Score linha a linha (informativo, não consolidado)

In [6]:
df["score_trafego_2025"] = df["renda_atualizada_2025"] * df["Total_0_9"]

# 6. Consolidar por CEP (SEM SOMAR — usar média!)

In [7]:
df_cep = (
    df.groupby("CEP", as_index=False)
      .agg({
          "Bairro": lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0],
          "renda_atualizada_2025": "median",
          "Total_0_9": "median",
          "populacao_total": "median"
      })
)

In [8]:
# Renomear colunas para refletir que são médias, não totais
df_cep = df_cep.rename(columns={
    "renda_atualizada_2025": "renda_mediana_2025",
    "Total_0_9": "mediana_criancas_0_9",
    "populacao_total": "populacao_mediana"
})

# 7. Score final no nível do CEP (correto)

In [9]:
df_cep["score_trafego_2025"] = (
    df_cep["renda_mediana_2025"] * df_cep["mediana_criancas_0_9"]
)

# 8. Ranking final

In [10]:
top_ceps = df_cep.sort_values("score_trafego_2025", ascending=False)

print(top_ceps.head(15))

          CEP             Bairro  renda_mediana_2025  mediana_criancas_0_9  \
0   04561-004  Brooklin Paulista                 NaN                  39.0   
1   04561-050  Brooklin Paulista                 NaN                  20.0   
2   04561-070  Brooklin Paulista                 NaN                  11.0   
3   04562-030  Brooklin Paulista                 NaN                  31.0   
4   04562-040  Brooklin Paulista                 NaN                  43.0   
5   04562-080  Brooklin Paulista                 NaN                  35.0   
6   04563-004     Cidade Monções                 NaN                  16.0   
7   04563-013     Cidade Monções                 NaN                  19.0   
8   04563-060     Cidade Monções                 NaN                  27.0   
9   04563-061     Cidade Monções                 NaN                  30.0   
10  04563-908     Cidade Monções                 NaN                  48.0   
11  04564-003     Cidade Monções                 NaN            

In [11]:
# Arredondar para 2 casas decimais (padrão monetário)
top_ceps["renda_mediana_2025"] = top_ceps["renda_mediana_2025"].round(2)
top_ceps["score_trafego_2025"] = top_ceps["score_trafego_2025"].round(2)

In [12]:
# Salvar resultado agregado na pasta do notebook (comportamento original)
top_ceps.to_csv("morumbi_top_ceps_2025.csv", index=False)

In [13]:
# Lista dos bairros desejados
bairros_desejados = ["Vila Sônia", "Ferreira", "Paraisópolis", "Jardim Maria Duarte", "Vila Andrade"]

# Filtrar diretamente no top_ceps
top_ceps_filtrado = top_ceps[top_ceps["Bairro"].isin(bairros_desejados)]

top_ceps_filtrado

,CEP,Bairro,renda_mediana_2025,mediana_criancas_0_9,populacao_mediana,score_trafego_2025
215,05520-200,Vila Sônia,NaN,59.0,553.0,NaN
216,05520-400,Vila Sônia,NaN,31.5,365.5,NaN
217,05521-100,Vila Sônia,NaN,10.0,119.0,NaN
218,05521-200,Vila Sônia,NaN,48.0,385.5,NaN
219,05522-030,Ferreira,NaN,20.0,272.0,NaN
...,...,...,...,...,...,...
804,05752-440,Jardim Maria Duarte,NaN,64.0,624.0,NaN
806,05752-520,Jardim Maria Duarte,NaN,35.5,490.0,NaN
807,05752-540,Jardim Maria Duarte,NaN,52.5,497.5,NaN
808,05752-570,Jardim Maria Duarte,NaN,32.0,346.0,NaN


In [14]:
top_ceps_filtrado.to_csv("morumbi_top_ceps_filtrados_2025.csv", index=False)